In [4]:
import pandas as pd
import numpy as np
import os

In [5]:

def get_random_partitions(dataset, num_partitions):

    ds_random = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    indices = ds_random.index
    split_indices = np.array_split(indices, num_partitions)
    return [ds_random.loc[idx] for idx in split_indices]

   

def create_federated_dataset_random(num_clients=5):
    """
    With write=True, this function writes the dataset, otherwise it simply 
    """
    dataset = pd.read_csv('../data/cox-violent-parsed_filt.csv')
    os.makedirs('../federated_data_random', exist_ok=True)

    data_partitions = get_random_partitions(dataset,num_clients)

    for i,partition in enumerate(data_partitions):
        
        if(i == 0):
            partition.to_csv(f'../federated_data_random/client_random_test.csv', index=False) # type: ignore
            continue

        partition.to_csv(f'../federated_data_random/client_random_{i+1}.csv', index=False) # type: ignore

    return data_partitions

def create_federated_dataset_sensitive(num_clients=5, alpha=0.5):
    """
    Split centralized dataset into multiple client datasets with different data distributions
    using Dirichlet distribution for non-IID partitioning and write data in CSV files (overwrite).
    
    Parameters:
    -----------
    dataset : pd.DataFrame
        Centralized dataset containing a 'race' column
    num_clients : int
        Number of clients to split data across
    alpha : float
        Dirichlet concentration parameter (lower = more skewed, higher = more uniform)
    output_dir : str
        Directory to save client datasets
        
    Returns:
    --------
    client_indices : list of lists
        Indices assigned to each client
    """
    # Create output directory
    output_dir = '../federated_data_sensitive'
    os.makedirs(output_dir, exist_ok=True)
    dataset = pd.read_csv('../data/cox-violent-parsed_filt.csv')

    # Validate inputs
    if 'race' not in dataset.columns:
        raise ValueError("Dataset must contain a 'race' column")
    if num_clients < 1:
        raise ValueError("num_clients must be at least 1")
    if alpha <= 0:
        raise ValueError("alpha must be positive")

    # Partition the dataset and extract the test set with 25% test set
    data_partitions = get_random_partitions(dataset, num_partitions=4)
    test_set = data_partitions[0]
    train_pool = pd.concat(data_partitions[1:]).reset_index(drop=True)

    
    test_set.to_csv(f'{output_dir}/client_sensitive_test.csv', index=False)

    # Get unique categories
    categories = dataset['race'].unique()
    client_indices = [[] for _ in range(num_clients)]



    # Distribute each category across clients using Dirichlet distribution
    for cat in categories:
        # Get all indices for this category
        idx_cat = dataset[dataset['race'] == cat].index.values
        np.random.shuffle(idx_cat)

        # Dirichlet distribution determines how many samples of this race go to each client
        # Low alpha = high skew; High alpha = more uniform
        proportions = np.random.dirichlet([alpha] * num_clients)

        # Convert proportions to split points
        proportions = (np.cumsum(proportions) * len(idx_cat)).astype(int)[:-1]
        split_idx = np.split(idx_cat, proportions)

        # Assign splits to clients
        for i in range(num_clients):
            client_indices[i].extend(split_idx[i])

    # Save each client's dataset
    for i in range(num_clients):
        # Shuffle local data
        client_df = dataset.iloc[client_indices[i]].sample(frac=1, random_state=42)

        # Save to CSV
        output_path = os.path.join(output_dir, f'client_skewed_{i+1}.csv')
        client_df.to_csv(output_path, index=False)

        # Print statistics
        print(f"Client {i+1} size: {len(client_df)} | Race distribution:")
        print(client_df['race'].value_counts(normalize=True))
        print()

    return client_indices

severity_rank = {
    "(X)": 1,
    "(NI0)": 2,
    "(CT)": 3,
    "(M03)": 4,
    "(M2)": 5,
    "(M1)": 6,
    "(CO3)": 7,
    "(TCX)": 8,
    "(F7)": 9,
    "(F6)": 10,
    "(F5)": 11,
    "(F3)": 12,
    "(F2)": 13,
    "(F1)": 14
}

def prepare_dataset(data_violent_filt):

    data_cleaned = data_violent_filt[['decile_score', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'c_jail_out', 'c_charge_degree', 'c_jail_in']].copy()
    data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
    data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
    data_cleaned['jail_time'] = (data_cleaned['c_jail_out'] - data_cleaned['c_jail_in']).dt.total_seconds() / (3600)
    data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)
    # list(data_cleaned['c_charge_degree'].unique())
    data_cleaned = data_cleaned.map(lambda x: severity_rank.get(x) if x in severity_rank else x)
    data_cleaned.dropna(inplace=True)
    return data_cleaned


#### Centralized Dataset

In [6]:
data_violent_filt = pd.read_csv('../data/cox-violent-parsed_filt.csv')
data_prepared = prepare_dataset(data_violent_filt)
data_prepared.to_csv('../data/centralized_dataset.csv', index=False)


#### Decenralized Dataset

In [7]:
create_federated_dataset_random()
create_federated_dataset_sensitive()

Client 1 size: 4771 | Race distribution:
race
Caucasian           0.465730
African-American    0.309998
Hispanic            0.213792
Asian               0.005869
Native American     0.003773
Other               0.000838
Name: proportion, dtype: float64

Client 2 size: 712 | Race distribution:
race
Caucasian           0.382022
Other               0.286517
African-American    0.252809
Native American     0.047753
Hispanic            0.030899
Name: proportion, dtype: float64

Client 3 size: 1648 | Race distribution:
race
Caucasian          0.741505
Other              0.205704
Hispanic           0.035194
Asian              0.015170
Native American    0.002427
Name: proportion, dtype: float64

Client 4 size: 4025 | Race distribution:
race
Caucasian           0.563975
African-American    0.303851
Other               0.077516
Hispanic            0.050683
Asian               0.003975
Name: proportion, dtype: float64

Client 5 size: 7160 | Race distribution:
race
African-American    0.964944
Hi

[[np.int64(2218),
  np.int64(10108),
  np.int64(3810),
  np.int64(3765),
  np.int64(9218),
  np.int64(16645),
  np.int64(13790),
  np.int64(4435),
  np.int64(2661),
  np.int64(15164),
  np.int64(13394),
  np.int64(8671),
  np.int64(1775),
  np.int64(6918),
  np.int64(6015),
  np.int64(13256),
  np.int64(11972),
  np.int64(15430),
  np.int64(2062),
  np.int64(599),
  np.int64(15625),
  np.int64(11519),
  np.int64(5283),
  np.int64(9502),
  np.int64(8519),
  np.int64(3675),
  np.int64(6555),
  np.int64(12966),
  np.int64(11264),
  np.int64(105),
  np.int64(8668),
  np.int64(12378),
  np.int64(11551),
  np.int64(17412),
  np.int64(5863),
  np.int64(8853),
  np.int64(14666),
  np.int64(16784),
  np.int64(5782),
  np.int64(8421),
  np.int64(13729),
  np.int64(14443),
  np.int64(2466),
  np.int64(6796),
  np.int64(7905),
  np.int64(13077),
  np.int64(10026),
  np.int64(12198),
  np.int64(5290),
  np.int64(16572),
  np.int64(14628),
  np.int64(5501),
  np.int64(5210),
  np.int64(4693),
  np.i

In [5]:
from get_datasets import get_federated_datasets_random, get_federated_datasets_sensitive, get_centralized_dataset

federated_random = get_federated_datasets_random()
federated_sensitive = get_federated_datasets_sensitive()
centralized_dataset = get_centralized_dataset()

federated_random[0]

,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,...,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,8242.0,dylan welly,dylan,welly,Male,18/04/1994,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,4,Low,16/03/2013,Risk of Violence,5,Medium,0,0
1,6933.0,guillermo laffiteau,guillermo,laffiteau,Male,23/05/1957,58,Greater than 45,Hispanic,0,...,NaN,Risk of Recidivism,1,Low,20/06/2014,Risk of Violence,1,Low,4,0
2,9844.0,luis gonzalez,luis,gonzalez,Male,22/05/1992,23,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,26/12/2013,Risk of Violence,4,Low,0,0
3,NaN,joseph ambers,joseph,ambers,Male,17/09/1966,49,Greater than 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,02/08/2014,Risk of Violence,1,Low,6,0
4,NaN,anthony nicholasi,anthony,nicholasi,Male,15/11/1993,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,01/07/2014,Risk of Violence,4,Low,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3659,3849.0,jerrod moore,jerrod,moore,Male,15/11/1983,32,25 - 45,African-American,0,...,NaN,Risk of Recidivism,8,High,15/01/2013,Risk of Violence,6,Medium,12,0
3660,10599.0,manuel aguasvivas,manuel,aguasvivas,Male,06/12/1988,27,25 - 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,17/09/2014,Risk of Violence,6,Medium,2,0
3661,6226.0,lawrence gaston,lawrence,gaston,Male,08/04/1993,23,Less than 25,African-American,0,...,NaN,Risk of Recidivism,4,Low,26/08/2013,Risk of Violence,6,Medium,1,0
3662,NaN,eric mckenzie,eric,mckenzie,Male,15/03/1990,26,25 - 45,African-American,0,...,NaN,Risk of Recidivism,9,High,26/08/2013,Risk of Violence,10,High,0,0
